# Revenue Overview

Exploratory analysis of POD revenue across all source channels.

**Data source:** `marts.fct_revenue`  
**Last updated:** 2026-03-06

---

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv(dotenv_path='../../.env')

DATABASE_URL = os.environ['DATABASE_URL']
engine = create_engine(DATABASE_URL)

print('Connected to:', DATABASE_URL.split('@')[-1])  # hide credentials

## 1. Monthly Net Revenue by Source

In [ ]:
monthly_revenue = pd.read_sql(
    """
    select
        revenue_month,
        source,
        sum(net_revenue_usd) as net_revenue_usd
    from marts.fct_revenue
    where revenue_month >= date_trunc('month', current_date) - interval '12 months'
    group by 1, 2
    order by 1, 2
    """,
    engine,
    parse_dates=['revenue_month'],
)

monthly_revenue.head(10)

In [ ]:
pivot = monthly_revenue.pivot_table(
    index='revenue_month', columns='source', values='net_revenue_usd', aggfunc='sum'
).fillna(0)

fig, ax = plt.subplots(figsize=(14, 5))
pivot.plot(kind='bar', stacked=True, ax=ax, colormap='tab10')

ax.set_title('Monthly Net Revenue by Source (Last 12 Months)', fontsize=14)
ax.set_xlabel('')
ax.set_ylabel('Net Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.tick_params(axis='x', rotation=45)
ax.legend(title='Source', bbox_to_anchor=(1.01, 1), loc='upper left')

plt.tight_layout()
plt.show()

## 2. YTD Revenue Summary

In [ ]:
ytd_summary = pd.read_sql(
    """
    select
        source,
        sum(gross_revenue_usd)  as gross_usd,
        sum(refunds_usd)        as refunds_usd,
        sum(net_revenue_usd)    as net_usd,
        round(
            100.0 * sum(refunds_usd) / nullif(sum(gross_revenue_usd), 0),
            2
        )                       as refund_rate_pct
    from marts.fct_revenue
    where revenue_date >= date_trunc('year', current_date)
    group by 1
    order by net_usd desc
    """,
    engine,
)

ytd_summary.style.format({
    'gross_usd': '${:,.2f}',
    'refunds_usd': '${:,.2f}',
    'net_usd': '${:,.2f}',
    'refund_rate_pct': '{:.2f}%',
})

## 3. Daily Revenue Trend (Last 90 Days)

In [ ]:
daily = pd.read_sql(
    """
    select
        revenue_date,
        sum(net_revenue_usd) as net_revenue_usd
    from marts.fct_revenue
    where revenue_date >= current_date - interval '90 days'
    group by 1
    order by 1
    """,
    engine,
    parse_dates=['revenue_date'],
)

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(daily['revenue_date'], daily['net_revenue_usd'], alpha=0.3, color='steelblue')
ax.plot(daily['revenue_date'], daily['net_revenue_usd'], color='steelblue', linewidth=1.5)

# 7-day rolling average
rolling = daily['net_revenue_usd'].rolling(7).mean()
ax.plot(daily['revenue_date'], rolling, color='navy', linewidth=2, label='7-day avg')

ax.set_title('Daily Net Revenue — Last 90 Days', fontsize=14)
ax.set_ylabel('Net Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()